In [2]:
import pandas as pd
from pathlib import Path

In [ ]:
DATA_DIR = Path.cwd().parent / "data"
DATA_FILE = DATA_DIR / "flights_by_carrier_day_hour.csv.gz"
SUBMISSION_DIR = Path.cwd().parent / "submission"
SUBMISSION_HOURLY_DELAY = SUBMISSION_DIR / "hourly_delay_rates.csv"
SUBMISSION_CARRIER_ADJUSTED_DELAYS = SUBMISSION_DIR / "carrier_adjusted_delays.csv"

In [4]:
datos = pd.read_csv(DATA_FILE, compression="gzip")
datos

,year,month,day_of_week,scheduled_departure_hour,reporting_airline,scheduled_flights,cancelled_flights,operated_flights,delayed_departure_15_flights,positive_departure_delay_minutes
0,2006,1,1,0,AA,17.0,0.0,17.0,4.0,292.0
1,2006,1,1,0,AS,5.0,0.0,5.0,0.0,0.0
2,2006,1,1,0,B6,25.0,0.0,25.0,9.0,384.0
3,2006,1,1,0,CO,27.0,0.0,27.0,7.0,327.0
4,2006,1,1,0,DL,20.0,0.0,20.0,9.0,657.0
...,...,...,...,...,...,...,...,...,...,...
96066,2008,12,7,23,FL,5.0,0.0,5.0,0.0,4.0
96067,2008,12,7,23,MQ,1.0,0.0,1.0,1.0,587.0
96068,2008,12,7,23,NW,11.0,1.0,10.0,1.0,115.0
96069,2008,12,7,23,UA,43.0,2.0,41.0,12.0,2464.0


In [ ]:
agrupacion_x_hora_demora = (datos.groupby("scheduled_departure_hour", as_index=False)                             # Se agrupa por scheduled_departure_hour como indica el enunciado
                            [["operated_flights", "delayed_departure_15_flights"]].sum()                          # Se toman las dos columnas requeridas para calcular el índice y se suma cada agrupación generada para tener los datos listos para generar el índice

)

agrupacion_x_aerolinea_y_hora = (datos.groupby(["reporting_airline", "scheduled_departure_hour"], as_index=False) # Se agrupa por aerolínea y hora
                          [["operated_flights", "delayed_departure_15_flights"]].sum()
)

agrupacion_x_hora_demora["delay_rate"] = (agrupacion_x_hora_demora["delayed_departure_15_flights"] / agrupacion_x_hora_demora["operated_flights"])                                            # Se calcula la TND agrupando por hora
agrupacion_x_aerolinea_y_hora_tnd = pd.merge(agrupacion_x_aerolinea_y_hora, agrupacion_x_hora_demora[["delay_rate", "scheduled_departure_hour"]], on="scheduled_departure_hour", how="left")  # Se hace un merge para tener la TND por hora en el dataframe de aerolíneas y horas
agrupacion_x_aerolinea_y_hora_tnd["expected_delayed_flights"] = (agrupacion_x_aerolinea_y_hora_tnd["delay_rate"] * agrupacion_x_aerolinea_y_hora_tnd["operated_flights"])                                   # Se calcula la demora esperada por aerolínea y hora


agrupacion_aerolinea_tnd = (agrupacion_x_aerolinea_y_hora_tnd.groupby("reporting_airline", as_index=False)                                     # Se agrupa nuevamente por aerolínea dado que en agrupacion_x_aerolinea_y_hora_tnd se agrupó por hora y aerolínea, dando más de una observación para cada aerolínea. Se requiere una sola observación por aerolínea.
                  [["operated_flights", "delayed_departure_15_flights", "expected_delayed_flights"]].sum()
)

datos_filtrados = agrupacion_aerolinea_tnd.copy()
datos_filtrados = datos_filtrados[datos_filtrados["operated_flights"] >= 100000]



datos_filtrados["observed_to_expected_ratio"] = (datos_filtrados["delayed_departure_15_flights"] / datos_filtrados["expected_delayed_flights"])  # Se calcula la demora observada vs esperada por aerolínea
datos_filtrados["delay_rate"] = datos_filtrados["delayed_departure_15_flights"] / datos_filtrados["operated_flights"]
datos_filtrados["crude_rank"] = datos_filtrados["delay_rate"].rank(ascending=False).astype(int)                   # ascending=False para ir de mayor a menor delay, el ranking en enteros para seguir las instrucciones del enunciado
datos_filtrados["adjusted_rank"] = datos_filtrados["observed_to_expected_ratio"].rank(ascending=False).astype(int)  # ascending=False para ir de mayor a menor razón, el ranking en enteros para seguir las instrucciones del enunciado
datos_filtrados = datos_filtrados.sort_values("adjusted_rank").reset_index(drop=True)

In [ ]:
# ORDENAR COLUMNAS HOURLY DELAY RATES
HOURLY_DELAY_RATES_COLUMNS = ["scheduled_departure_hour", "operated_flights", "delayed_departure_15_flights", "delay_rate"]
CARRIER_ADJUSTED_DELAYS_COLUMNS = ["reporting_airline", "operated_flights", "delayed_departure_15_flights", "delay_rate",
                                   "expected_delayed_flights", "observed_to_expected_ratio", "crude_rank", "adjusted_rank"]
agrupacion_x_hora_demora = agrupacion_x_hora_demora[HOURLY_DELAY_RATES_COLUMNS]
datos_filtrados = datos_filtrados[CARRIER_ADJUSTED_DELAYS_COLUMNS]

with open(SUBMISSION_HOURLY_DELAY, "w") as f:
    agrupacion_x_hora_demora.to_csv(f, index=False)

with open(SUBMISSION_CARRIER_ADJUSTED_DELAYS, "w") as f:
    datos_filtrados.to_csv(f, index=False)